# Sibling Scene Maker — architecture walkthrough & live demo

**Category:** Personal Expression &nbsp;|&nbsp; **Lane:** Community

**Live app:** https://7qd9l32i01.execute-api.us-east-1.amazonaws.com/

This notebook drives the actual deployed AWS pipeline end to end from Python: it calls the live API, waits for AWS Lambda to render a real clip with Remotion + AWS Polly voices, and plays the result inline. It also calls AWS Polly directly, independent of the app, to demonstrate the underlying AWS service on its own.

No AWS credentials are required for the first two sections — they just call the public HTTPS API. The last section (direct Polly call) needs `boto3` configured with your own AWS credentials if you want to run it yourself.

## 1. Submit a scene to the live API

`API_BASE` is the app's own HTTPS endpoint (API Gateway → AWS Lambda). We send either structured `lines` or a free-form `script`; the backend infers mood/location/activity/music from the script text when it's not set explicitly.

In [1]:
import requests

API_BASE = "https://7qd9l32i01.execute-api.us-east-1.amazonaws.com"

payload = {
    "script": (
        "Elahi: Let's dance on stage tonight!\n"
        "Sarvgun: I love dancing with you, sister.\n"
        "We both start dancing to pop music."
    )
}

resp = requests.post(f"{API_BASE}/generate", json=payload, timeout=30)
resp.raise_for_status()
result = resp.json()
result

{'renderId': '9s2ecswqi3',
 'bucketName': 'remotionlambda-useast1-lxwftvy1r4',
 'functionName': 'remotion-render-4-0-523-mem2048mb-disk2048mb-120sec',
 'region': 'us-east-1',
 'understood': {'lines': [{'speaker': 'elahi',
    'text': "Let's dance on stage tonight!"},
   {'speaker': 'sarvgun', 'text': 'I love dancing with you, sister.'},
   {'speaker': 'elahi', 'text': 'We both start dancing to pop music.'}],
  'mood': 'warm',
  'location': 'stage',
  'activity': 'dancing',
  'music': 'pop'}}

## 2. Poll AWS Lambda until the render finishes

The render itself runs on a separate Remotion Lambda function, compositing the animated scene with the Polly audio into an MP4 and writing it to S3. This polls the status endpoint (which in turn checks Remotion Lambda's own progress) until it's done, then plays the result.

In [2]:
import time
from IPython.display import Video, display

render_id = result["renderId"]
bucket_name = result["bucketName"]

output_file = None
for _ in range(60):
    time.sleep(2.5)
    status = requests.get(
        f"{API_BASE}/status/{render_id}",
        params={"bucketName": bucket_name},
        timeout=30,
    ).json()
    pct = round((status.get("overallProgress") or 0) * 100)
    print(f"Rendering on AWS Lambda... {pct}%")
    if status.get("fatalErrorEncountered"):
        raise RuntimeError(status.get("errors"))
    if status.get("done"):
        output_file = status["outputFile"]
        break

print("Done:", output_file)
display(Video(output_file, embed=False))

Rendering on AWS Lambda... 22%


Rendering on AWS Lambda... 35%


Rendering on AWS Lambda... 49%


Rendering on AWS Lambda... 62%


Rendering on AWS Lambda... 74%


Rendering on AWS Lambda... 82%


Rendering on AWS Lambda... 86%


Rendering on AWS Lambda... 100%
Done: https://s3.us-east-1.amazonaws.com/remotionlambda-useast1-lxwftvy1r4/renders/9s2ecswqi3/out.mp4


## 3. AWS Polly on its own

The two sections above call the app's API, which calls Polly internally. This cell calls **Amazon Polly directly** with `boto3`, independent of the app, to show the underlying AWS AI service by itself — the same call the backend Lambda makes for every line of dialogue. Requires your own AWS credentials configured (`aws configure` or environment variables) with `polly:SynthesizeSpeech` permission.

In [3]:
import boto3
from IPython.display import Audio

polly = boto3.client("polly", region_name="us-east-1")

response = polly.synthesize_speech(
    Engine="standard",
    OutputFormat="mp3",
    VoiceId="Ivy",  # Elahi's voice in the app
    Text="Your side ends at the couch.",
)

audio_bytes = response["AudioStream"].read()
with open("polly_sample.mp3", "wb") as f:
    f.write(audio_bytes)

Audio("polly_sample.mp3")

## Architecture summary

| Piece | AWS service | Role |
|---|---|---|
| Voices | Amazon Polly | One synthesized voice per character |
| Orchestration | AWS Lambda (`generate`, `status`) | Polly → S3 → triggers a render; polls progress |
| Rendering | AWS Lambda (Remotion Lambda) | Composites animation + audio into an MP4 |
| Storage | Amazon S3 | Site bundle, generated audio, rendered videos |
| API | Amazon API Gateway (HTTP API) | Public HTTPS endpoints, throttled |
| Frontend | AWS Lambda + API Gateway | Serves the web UI over the same HTTPS domain |
| Cost control | AWS Budgets | $5/month alarm |

Full source, including the SAM infrastructure template and the Remotion animation code: https://github.com/Rajpreet12/Sibling-Scene-Maker